# Ngày 7 · Dự án House Prices

- Hoàn thiện pipeline có baseline và validation
- Xuất submission CSV theo Id và SalePrice

**Đọc:**
- ISLR, §3.3 và §6.2
- ISLR, §5.1.3–5.1.4

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge

DATA = next((p for p in (Path('data'), Path('../data')) if (p/'real_estate.csv').is_file()), None)
if DATA is None:
    raise FileNotFoundError('Mở notebook mẫu từ thư mục py30 hoặc bai_lam.')
real_estate = pd.read_csv(DATA / 'real_estate.csv')

def _check(name, fn):
    try:
        ok, detail = fn()
    except NotImplementedError:
        ok, detail = False, 'Chưa cài đặt (NotImplementedError)'
    except Exception as error:
        ok, detail = False, f'{type(error).__name__}: {error}'
    return {'name': name, 'ok': bool(ok), 'detail': str(detail)}

def _report(ex_id, *checks):
    print('__PY30__' + json.dumps({'id': ex_id, 'checks': list(checks)}, ensure_ascii=False))
def _test_pipeline(fn, output):
    X=np.arange(30.,dtype=float).reshape(15,2)
    X[3,0]=np.nan
    y=np.arange(15.)*100+10000
    test=np.array([[2.,3.],[10.,np.nan]])
    ids=[77,55]
    result=fn(X,y,test,ids,seed=7)
    if not output:
        train=np.random.default_rng(7).permutation(len(y))[:12]
        sample=X[train]
        expected=np.where(np.isnan(sample),np.nanmedian(sample,axis=0),sample).mean(axis=0)
        return np.allclose(result['mu'],expected) and np.isfinite(result['mae']) and np.isfinite(result['baseline_mae'])
    repeat=fn(X,y,test,ids,seed=7)
    sub=result['submission']
    return list(sub.columns)==['Id','SalePrice'] and sub.Id.tolist()==ids and np.isfinite(sub.SalePrice).all() and sub.equals(repeat['submission'])
def export_house_prices(run_pipeline, lam=1.0):
    folder=DATA/'kaggle/house-prices-advanced-regression-techniques'
    train=pd.read_csv(folder/'train.csv')
    test=pd.read_csv(folder/'test.csv')
    columns=[c for c in train.select_dtypes(include='number') if c not in ('Id','SalePrice') and c in test.columns]
    result=run_pipeline(train[columns].to_numpy(float),train.SalePrice.to_numpy(float),test[columns].to_numpy(float),test.Id,lam=lam)
    result['submission'].to_csv('submission.csv',index=False)
    return result['mae'], result['baseline_mae']


## Pipeline Ridge và tệp Kaggle

Gợi ý: Theo đúng quy ước chia trong ô code. Viết một hàm fit(features, target) dùng chung cho validation và lần fit cuối trên toàn bộ train.csv. Chạy export_house_prices(run_pipeline) sau khi tải dữ liệu Kaggle để ghi submission.csv.

In [ ]:
def run_pipeline(X, y, test_X, ids, seed=42, lam=1.0):
    # Quy ước chia: order = np.random.default_rng(seed).permutation(len(y));
    # 80% đầu (int(0.8*n)) là train, phần còn lại là validation.
    # Điền giá trị thiếu bằng median của train, chuẩn hóa bằng mean/std của train.
    # Trả dict: mu (mean sau khi điền thiếu, tính trên train), mae, baseline_mae (median train), submission (DataFrame Id, SalePrice).
    raise NotImplementedError


In [ ]:
_report('d7-pipeline',
    _check('Fit thống kê theo train', lambda: (_test_pipeline(run_pipeline, False), 'Kiểm chứng mu với phân hoạch theo seed.')),
    _check('Tái lập và định dạng submission', lambda: (_test_pipeline(run_pipeline, True), 'Kiểm tra đúng Id, cột, thứ tự và số dự đoán hữu hạn.')),
)


## Dự án

Tải House Prices sau khi chấp nhận luật cuộc thi. Làm pipeline trên train.csv và test.csv, chọn cột số và λ. Kiểm tra submission.csv giữ đúng Id, SalePrice và thứ tự test trước khi nộp thủ công.